# Vector Databases: ChromaDB, Semantic Search & Text Chunking

> **Focus Area:** ভেক্টর ডেটাবেস (Vector Databases) ও RAG-এর Retrieval ভিত্তি
> **Topics:** Brute-force বনাম ANN; ChromaDB (collection, metadata, filter); Semantic বনাম Keyword Search; Chunking (fixed, recursive, overlap)
> **Achievement:** একটা **Document Indexer**-এর বিল্ডিং ব্লকগুলো হাতে-কলমে দেখা — ডকুমেন্ট কীভাবে চাংকে ভাগ হয়, ভেক্টর হয়ে ChromaDB-তে বসে, আর প্রশ্ন করলে প্রাসঙ্গিক চাংক ফেরত আসে।

---

## 1. Topic: Vector Databases — ChromaDB, Semantic Search, and Text Chunking

Week 10-এ আমরা টেক্সটকে embedding ভেক্টরে বদলেছি আর NumPy দিয়ে cosine similarity মেপেছি। কিন্তু ৫ লাখ চাংকের প্রতিটার সাথে প্রতি প্রশ্নে তুলনা করা ধীর। **Vector Database** এই কাজ দ্রুত ও গুছিয়ে করে। এই নোটবুকে:

* **Brute-force বনাম ANN** — কেন সব ভেক্টর স্ক্যান করা scale করে না।
* **ChromaDB** — সার্ভার ছাড়া লোকাল ভেক্টর ডেটাবেস: `add`, `query`, metadata filter।
* **Semantic vs Keyword Search** — অর্থ দিয়ে খোঁজা বনাম হুবহু শব্দ দিয়ে খোঁজা।
* **Chunking** — fixed-size, recursive, overlap এবং chunk size-এর trade-off।

নোটবুকের বেশিরভাগ cell **অফলাইনে, ছোট toy data দিয়ে** চলে। যে cell আসল embedding মডেল (`paraphrase-MiniLM-L3-v2`) লাগে, সেটা `RUN_MODEL = True` হলে চলে।

---

## 2. Why It Is Related

পরের ক্লাসে আমরা RAG বট বানাব: (১) প্রাসঙ্গিক তথ্য খুঁজে আনা (**Retrieval**), (২) সেই তথ্য দিয়ে LLM-কে উত্তর লেখানো (**Generation**)। Retrieval দুর্বল হলে LLM ভুল বা অপ্রাসঙ্গিক context পায়, আর hallucinate করে। তাই আজকের বিষয় RAG-এর সবচেয়ে গুরুত্বপূর্ণ ভিত্তি।

```text
Document → Chunking (+Overlap) → Embedding → ChromaDB
User Query → Embedding → Similarity Search → Top-k Chunks → (পরের ক্লাসে) LLM
```

---

## 3. How It Works

### 3.1 Brute-force কেন যথেষ্ট না?

Brute-force মানে query ভেক্টরকে **প্রতিটা** সংরক্ষিত ভেক্টরের সাথে মেলানো — খরচ `O(N × d)`। Vector database সাধারণত **ANN (Approximate Nearest Neighbor)** index (যেমন HNSW) ব্যবহার করে: সব ভেক্টর না দেখে ঠিক "প্রতিবেশী" এলাকায় গিয়ে খোঁজে। ফল প্রায় নিখুঁত, কিন্তু অনেক দ্রুত।

### হাতে-কলমে: brute-force-এর খরচ মাপা

১০০,০০০টা ৩৮৪-মাত্রার random ভেক্টর (MiniLM-এর সাইজ) বানিয়ে একটা query-র top-3 খুঁজব। এটা **N বাড়লে সময় কীভাবে বাড়ে** তা দেখার জন্য।

In [1]:
import time
import numpy as np

rng = np.random.default_rng(42)
dim = 384

def brute_force_top_k(matrix, query, k=3):
    # cosine similarity = normalize করে dot product
    m = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    q = query / np.linalg.norm(query)
    sims = m @ q
    top = np.argpartition(-sims, k)[:k]
    return top[np.argsort(-sims[top])], sims

for n in (1_000, 10_000, 100_000):
    matrix = rng.normal(size=(n, dim)).astype("float32")
    query = rng.normal(size=dim).astype("float32")
    start = time.perf_counter()
    top, sims = brute_force_top_k(matrix, query)
    ms = (time.perf_counter() - start) * 1000
    print(f"N={n:>7,}  top-3 index={top.tolist()}  time={ms:7.1f} ms")

assert len(top) == 3 and sims[top[0]] >= sims[top[1]] >= sims[top[2]]
print("Check: top-3 sorted; সময় N-এর সাথে বাড়ে — scale বাড়লে ANN index-এর দরকার।")

N=  1,000  top-3 index=[285, 674, 673]  time=    0.3 ms
N= 10,000  top-3 index=[814, 3317, 4402]  time=    1.8 ms


N=100,000  top-3 index=[57518, 31369, 12355]  time=   29.4 ms
Check: top-3 sorted; সময় N-এর সাথে বাড়ে — scale বাড়লে ANN index-এর দরকার।


**খেয়াল করুন:** N বাড়লে সময় বাড়ে (ছোট N-এ overhead বেশি, তাই অনুপাত নিখুঁত ১:১০০ না; আপনার মেশিনে সংখ্যা আলাদা হবে)। ১ লাখ ভেক্টরে ~দশ মিলিসেকেন্ড-ঘরের খরচ একটা query-তে সহনীয়, কিন্তু কয়েক মিলিয়ন ভেক্টর ও অনেক concurrent request-এ অচল। ANN accuracy-র সামান্য ছাড় দিয়ে এই খরচ কমায়।

---

### 3.2 ChromaDB Basics

`chromadb` তে মূল ধারণা চারটা: **client** (ডেটাবেস), **collection** (টেবিলের মতো), প্রতিটা item-এ `id + embedding + document + metadata`, আর `query()`। Metadata দিয়ে `where` filter করা যায় — পরে citation ও scope-limited search-এর ভিত্তি।

### হাতে-কলমে: toy ভেক্টর দিয়ে add, query ও filter

আসল মডেল ছাড়াই বোঝার জন্য আমরা হাতে ৩-মাত্রার ভেক্টর দিচ্ছি: `[leave, remote-work, money]`।

In [2]:
import chromadb

client = chromadb.EphemeralClient()   # মেমোরিতে; PersistentClient(path=...) দিলে ডিস্কে সেভ হয়
col = client.get_or_create_collection(
    name="toy_handbook",
    metadata={"hnsw:space": "cosine"},   # default L2; এখানে cosine distance চাই
)

items = [
    ("sick",   "Staff get 14 days of paid sick leave.",           [0.9, 0.0, 0.1], {"source": "handbook.pdf", "page": 14}),
    ("remote", "Remote work is allowed two days per week.",       [0.1, 0.9, 0.0], {"source": "handbook.pdf", "page": 27}),
    ("expense","Expense reports are due within 30 days.",         [0.0, 0.1, 0.9], {"source": "finance.pdf",  "page": 3}),
]
col.upsert(ids=[i[0] for i in items], documents=[i[1] for i in items],
           embeddings=[i[2] for i in items], metadatas=[i[3] for i in items])

res = col.query(query_embeddings=[[0.8, 0.1, 0.1]], n_results=2)
print("Top-2 for a 'leave-like' query:")
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"  dist={dist:.3f}  {meta['source']} p.{meta['page']}  {doc}")

filtered = col.query(query_embeddings=[[0.8, 0.1, 0.1]], n_results=3, where={"source": "finance.pdf"})
print("\nfinance.pdf-এর মধ্যে সীমিত:", filtered["ids"][0])

assert res["ids"][0][0] == "sick"
assert filtered["ids"][0] == ["expense"]
print("Check: nearest = sick leave chunk; where-filter অন্য source বাদ দেয়।")

Top-2 for a 'leave-like' query:
  dist=0.008  handbook.pdf p.14  Staff get 14 days of paid sick leave.
  dist=0.769  handbook.pdf p.27  Remote work is allowed two days per week.

finance.pdf-এর মধ্যে সীমিত: ['expense']
Check: nearest = sick leave chunk; where-filter অন্য source বাদ দেয়।


**খেয়াল করুন:** (১) `distance` যত **কম**, মিল তত **বেশি**। (২) Default metric L2; cosine চাইলে collection বানানোর সময় `hnsw:space` দিতে হয় — পরে বদলানো যায় না। (৩) `upsert` একই `id` দুবার চালালে duplicate হয় না; `add` একই id-তে সমস্যা করে — notebook re-run করার সময় এটা কাজে লাগে।

---

### 3.3 Semantic Search vs. Keyword Search

* **Keyword**: হুবহু শব্দ/substring মেলে। "medical absence" লিখলে "sick leave" পাবে না।
* **Semantic**: অর্থ মেলে। কিন্তু নির্দিষ্ট ID/কোড (যেমন `Policy #4471`) বোঝে না।
* **Hybrid**: দুটো একসাথে চালিয়ে ফল মেলানো — production-এ সাধারণ।

### হাতে-কলমে: একই query, দুই পদ্ধতি

এই cell আসল embedding মডেল লাগে (প্রথমবার ~৬১MB download)। মডেল না পেলে cell skip হবে।

In [3]:
RUN_MODEL = True

embed_model = None
if RUN_MODEL:
    try:
        from sentence_transformers import SentenceTransformer
        embed_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")
    except Exception as e:
        print("Model লোড হয়নি, model-ভিত্তিক cell skip হবে:", type(e).__name__)

docs = [
    "All full-time staff are entitled to fourteen days of paid sick leave per calendar year.",
    "Employees may work remotely up to two days per week with manager approval.",
    "Expense reports must be filed within thirty days of the expense date.",
    "Policy #4471 governs the use of company vehicles for personal errands.",
]

def keyword_search(query, docs):
    return [d for d in docs if query.lower() in d.lower()]

if embed_model:
    sem_col = client.get_or_create_collection("semantic_demo", metadata={"hnsw:space": "cosine"})
    sem_col.upsert(ids=[f"d{i}" for i in range(len(docs))], documents=docs,
                   embeddings=embed_model.encode(docs).tolist())

    def semantic_search(query, k=1):
        r = sem_col.query(query_embeddings=embed_model.encode([query]).tolist(), n_results=k)
        return r["documents"][0]

    for q in ["medical absence", "Policy #4471"]:
        print(f"Query: {q!r}")
        print("  keyword :", keyword_search(q, docs))
        print("  semantic:", semantic_search(q))
        print()

/Users/salmansakib/Documents/Projects/interview-Prep-Case-Study/Job Ready AI Bootcamp/week 11/Class 1 Project/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 34140.41it/s]

Query: 'medical absence'
  keyword : []
  semantic: ['All full-time staff are entitled to fourteen days of paid sick leave per calendar year.']

Query: 'Policy #4471'
  keyword : ['Policy #4471 governs the use of company vehicles for personal errands.']
  semantic: ['Policy #4471 governs the use of company vehicles for personal errands.']



**খেয়াল করুন:** `"medical absence"`-এ keyword খালি (`[]`), semantic sick-leave চাংক আনে — synonym-এ semantic জেতে। `"Policy #4471"`-এ এখানে দুটোই সঠিক, কারণ corpus মাত্র ৪টা চাংক। হাজার হাজার চাংকে অনেক অনুরূপ policy থাকলে `#4471` আর `#4417` embedding-এ প্রায় একই হয়ে যায়, তখন semantic ভুল করে আর keyword exact match দেয়। ছোট corpus-এর ফল দিয়ে সাধারণ সিদ্ধান্ত নেবেন না — তাই hybrid।

---

### 3.4 Text Chunking Strategies

পুরো ডকুমেন্ট এক ভেক্টরে রাখলে অর্থ "গড়পড়তা" হয়ে যায়, আর মডেলের input limit আছে। তাই ভাগ করি:

```text
Fixed-Size : প্রতি N ক্যারেক্টারে কাটো। সহজ, কিন্তু মাঝ-বাক্যে কাটে।
Recursive  : আগে \n\n (প্যারাগ্রাফ), তারপর বাক্য, তারপর শব্দ। অর্থ ভাঙে কম।
Overlap    : পরের চাংক আগের চাংকের শেষ অংশ দিয়ে শুরু হয়, বর্ডারে তথ্য হারায় না।
```

### হাতে-কলমে: বর্ডারে ভাঙা বাক্য ও overlap

একটা গুরুত্বপূর্ণ বাক্য ঠিক ২০০ ক্যারেক্টারের বর্ডারে ফেলে দেখি fixed-size তাকে ভাঙে কিনা, আর overlap ফেরত আনে কিনা। (কোনো মডেল লাগে না)

In [4]:
import math, re

filler = "Routine note about office hours and building access. " * 3
key_sentence = "Sick leave beyond two days requires a medical certificate."
text = (filler[:170] + key_sentence + " " + filler)   # key_sentence ১৭০-২৩০ ক্যারেক্টারে বসানো

def fixed_chunks(text, size):
    return [text[i:i + size] for i in range(0, len(text), size)]

def overlap_chunks(text, size, overlap):
    step = size - overlap        # প্রতিবার এতটুকু এগোই
    return [text[i:i + size] for i in range(0, len(text), step)]

def contains_whole(chunks, sentence):
    return any(sentence in c for c in chunks)

plain = fixed_chunks(text, 200)
lapped = overlap_chunks(text, 200, 60)

print(f"fixed   : {len(plain)} chunks, key sentence intact = {contains_whole(plain, key_sentence)}")
print(f"overlap : {len(lapped)} chunks, key sentence intact = {contains_whole(lapped, key_sentence)}")
print(f"expected overlap chunk count = ceil({len(text)}/{200-60}) = {math.ceil(len(text)/(200-60))}")

assert not contains_whole(plain, key_sentence)
assert contains_whole(lapped, key_sentence)
assert len(lapped) == math.ceil(len(text) / 140)
print("Check: overlap ছাড়া বাক্য ভাঙে; overlap-এ অন্তত একটা চাংকে সম্পূর্ণ থাকে।")

fixed   : 2 chunks, key sentence intact = False
overlap : 3 chunks, key sentence intact = True
expected overlap chunk count = ceil(377/140) = 3
Check: overlap ছাড়া বাক্য ভাঙে; overlap-এ অন্তত একটা চাংকে সম্পূর্ণ থাকে।


**খেয়াল করুন:** overlap-এর দাম আছে — চাংক সংখ্যা বাড়ে (`≈ len/(size−overlap)`), তাই storage ও embedding খরচ বাড়ে। ১০–১৫% overlap সাধারণত ভালো ভারসাম্য।

### হাতে-কলমে: recursive splitter (প্যারাগ্রাফ → বাক্য)

In [5]:
def recursive_chunks(text, target=250):
    out = []
    for para in (p.strip() for p in text.split("\n\n") if p.strip()):
        if len(para) <= target:
            out.append(para); continue
        buf = ""
        for sent in re.split(r"(?<=[.!?])\s+", para):
            if buf and len(buf) + 1 + len(sent) > target:
                out.append(buf); buf = sent
            else:
                buf = f"{buf} {sent}".strip()
        if buf: out.append(buf)
    return out

handbook = (
    "Leave Policy: Submit leave requests three business days ahead. Approved leave appears in payroll within one day. "
    "Unused leave up to five days carries forward.\n\n"
    "Sick Leave: Staff get fourteen paid sick days. A medical certificate is needed beyond two consecutive days.\n\n"
    "Remote Work: Up to two remote days per week with manager approval."
)
chunks = recursive_chunks(handbook, target=150)
for i, c in enumerate(chunks):
    print(f"[{i}] ({len(c):3d}) {c}")

assert all(c[-1] in ".!?" for c in chunks), "কোনো চাংক মাঝ-বাক্যে শেষ হওয়া উচিত না"
assert all(len(c) <= 150 or "." not in c[:-1] for c in chunks)
print("Check: প্রতিটা চাংক সম্পূর্ণ বাক্যে শেষ।")

[0] (112) Leave Policy: Submit leave requests three business days ahead. Approved leave appears in payroll within one day.
[1] ( 45) Unused leave up to five days carries forward.
[2] (107) Sick Leave: Staff get fourteen paid sick days. A medical certificate is needed beyond two consecutive days.
[3] ( 66) Remote Work: Up to two remote days per week with manager approval.
Check: প্রতিটা চাংক সম্পূর্ণ বাক্যে শেষ।


### হাতে-কলমে: chunk size বদলালে retrieval কেমন হয়?

একই ডকুমেন্ট ও একই query, চাংক সাইজ ১০০ / ৩০০ / ১৫০০। মডেল লাগে।

In [6]:
if embed_model:
    long_doc = "\n\n".join([
        "Leave Policy: Submit leave requests three business days ahead through the HR portal. Unused leave up to five days carries forward.",
        "Sick Leave: Full-time staff get fourteen days of paid sick leave per year. A medical certificate is needed beyond two consecutive days.",
        "Remote Work: Employees may work remotely two days per week with manager approval. Equipment is provided by IT.",
        "Expenses: Reports must be filed within thirty days. Receipts above fifty dollars are mandatory.",
    ])
    q = "how many paid sick days do I get"
    qv = embed_model.encode([q])
    for size in (100, 300, 1500):
        cs = [c for c in fixed_chunks(long_doc, size) if c.strip()]
        sims = (embed_model.encode(cs) @ qv.T).ravel()
        best = int(np.argmax(sims))
        print(f"size={size:>4}: {len(cs):>2} chunks | best len={len(cs[best]):>4} | text={cs[best][:70]!r}...")

size= 100:  5 chunks | best len= 100 | text=' to five days carries forward.\n\nSick Leave: Full-time staff get fourte'...
size= 300:  2 chunks | best len= 300 | text='Leave Policy: Submit leave requests three business days ahead through '...
size=1500:  1 chunks | best len= 476 | text='Leave Policy: Submit leave requests three business days ahead through '...


**খেয়াল করুন:** ১০০-এ চাংক মাঝ-বাক্যে শুরু/শেষ হয় (best চাংক অর্ধেক বাক্য); ১৫০০-এ পুরো ডকুমেন্ট এক চাংক, তাই retrieval-এর কোনো মানে থাকে না। ৩০০-এ fixed-size কাটা চাংকে একাধিক টপিক মিশে যায় এবং best চাংক sick-leave-এর বদলে leave-policy হতে পারে — **শুধু সাইজ ঠিক করলেই হয় না, কাটার জায়গাও ঠিক হতে হয়**; তাই recursive splitter দরকার।

---

## 4. Details & Valid Points

### 4.1 Vector Database Reference Table

| Database | Use Case | Advantage | Limitation |
| --- | --- | --- | --- |
| **ChromaDB** (এই কোর্সে) | লোকাল, ছোট-মাঝারি প্রজেক্ট | সার্ভার লাগে না, Python-native | কোটি ভেক্টরে সীমিত |
| FAISS | রিসার্চ, বড় স্কেল, in-memory | খুব দ্রুত, GPU সাপোর্ট | metadata/persistence নিজে বানাতে হয় |
| Pinecone | managed production | স্কেলেবল, অবকাঠামো ঝামেলাহীন | ক্লাউড, খরচ, ডেটা বাইরে যায় |
| Weaviate | hybrid search সহ production | শক্তিশালী filtering | সেটআপ জটিল |

> **Valid Point:** ChromaDB লোকাল ফাইলে সেভ হয়, তাই কোর্সের "Sovereign AI" ধারার সাথে মেলে। স্কেল বাড়লে অন্য ডেটাবেসে যাওয়া যুক্তিসঙ্গত; ধারণাগুলো (embedding, chunk, metadata, top-k) একই থাকে।

### 4.2 Chunk Size vs. Retrieval Quality

খুব ছোট চাংক অসম্পূর্ণ বাক্য হয়; খুব বড় চাংক একাধিক বিষয় মিশিয়ে সিগন্যাল পাতলা করে। প্রাথমিক সেটিং: **৩০০–৮০০ ক্যারেক্টার, ~১০–১৫% overlap** — তারপর নিজের ডেটায় মেপে ঠিক করুন।

### 4.3 কিছু বাস্তব সমস্যা

* **Embedding model মেলাতে হবে**: index ও query একই মডেলে embed না হলে distance অর্থহীন।
* **Re-run safety**: `upsert` ও stable ID ব্যবহার করুন, নইলে duplicate জমে।
* **Empty chunk** index করবেন না — অর্থহীন ভেক্টর noise তৈরি করে।
* **Distance ≠ probability**: `distance=0.4` মানে "৪০% মিল" না; শুধু তুলনার জন্য।

### 4.4 Analogy: Library Card Catalog

| Concept | Analogy |
| --- | --- |
| Brute-force | প্রতিটা বই খুলে পড়া |
| Vector DB (ANN) | বিষয়ভিত্তিক কার্ড ক্যাটালগ, সরাসরি সঠিক শেলফ |
| Chunking | অধ্যায়/পাতা আলাদা ইনডেক্স করা |
| Metadata | কার্ডে লেখক, বছর, শেলফ নম্বর |

---

## 5. Achievement: Document Indexer-এর পুরো flow

এবার জোড়া লাগাই: **text → chunk (overlap) → embed → ChromaDB (upsert + metadata) → query**। `embed` একটা **parameter**, তাই প্রথমে একটা নকল (stub) embedder দিয়ে pipeline-এর logic যাচাই করব — এটা **অর্থ বোঝে না**, শুধু শব্দ গুনে; তারপর আসল মডেল দিয়ে।

In [7]:
def stub_embed(texts, dim=64):
    # শব্দ-গণনা hash করে ভেক্টর; অর্থ বোঝে না, শুধু pipeline যাচাইয়ের জন্য
    out = []
    for t in texts:
        v = np.zeros(dim)
        for w in re.findall(r"[a-z0-9#]+", t.lower()):
            v[hash(w) % dim] += 1
        out.append(v.tolist())
    return out

def index_document(client, name, text, source, embed, size=300, overlap=45, chunker=None):
    col = client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})
    raw = chunker(text) if chunker else overlap_chunks(text, size, overlap)
    chunks = [c.strip() for c in raw if c.strip()]                                   # empty বাদ
    ids = [f"{source}:{i}" for i in range(len(chunks))]                              # stable id
    col.upsert(ids=ids, documents=chunks, embeddings=embed(chunks),
               metadatas=[{"source": source, "chunk_index": i} for i in range(len(chunks))])
    return col

def search(col, query, embed, k=2):
    r = col.query(query_embeddings=embed([query]), n_results=k)
    return list(zip(r["ids"][0], r["documents"][0], r["distances"][0]))

pipe_text = handbook * 2
col_a = index_document(client, "pipeline_stub", pipe_text, "handbook.txt", stub_embed)
n1 = col_a.count()
index_document(client, "pipeline_stub", pipe_text, "handbook.txt", stub_embed)   # re-run
print("chunks:", n1, "| after re-run:", col_a.count())

hit = search(col_a, "medical certificate sick days", stub_embed, k=1)[0]
print("top hit:", hit[0], "->", hit[1][:80], "...")

assert col_a.count() == n1, "re-run duplicate বানিয়েছে"
assert "sick" in hit[1].lower()
print("Check: re-run-এ duplicate নেই; sick query sick-leave চাংক আনে।")

chunks: 3 | after re-run: 3
top hit: handbook.txt:2 -> ff get fourteen paid sick days. A medical certificate is needed beyond two conse ...
Check: re-run-এ duplicate নেই; sick query sick-leave চাংক আনে।


In [8]:
if embed_model:
    embed_real = lambda texts: embed_model.encode(texts).tolist()
    col_b = index_document(client, "pipeline_real", handbook, "handbook.txt", embed_real,
                          chunker=lambda t: recursive_chunks(t, target=150))
    for q in ["How many days can I work from home?", "medical absence rules"]:
        _, doc, dist = search(col_b, q, embed_real, k=1)[0]
        print(f"Q: {q}\n  dist={dist:.3f}  {doc[:90]!r}\n")

Q: How many days can I work from home?
  dist=0.590  'Remote Work: Up to two remote days per week with manager approval.'

Q: medical absence rules
  dist=0.548  'Sick Leave: Staff get fourteen paid sick days. A medical certificate is needed beyond two '



**যাচাই:** আসল মডেলে "work from home" → Remote Work চাংক, "medical absence rules" → Sick Leave চাংক আসার কথা। না এলে কোন ধাপ দায়ী তা আলাদা করুন: chunker (বাক্য কাটা), embedding মডেল, নাকি query। stub embedder শুধু pipeline-এর logic (id, upsert, count) যাচাই করে, অর্থ না।

---

## 6. Summary ও নিজে পরীক্ষা

- Brute-force `O(N×d)`; vector DB ANN index দিয়ে এই খরচ কমায়, সামান্য accuracy ছেড়ে।
- ChromaDB: `client → collection → upsert(id, embedding, document, metadata) → query`; `where` দিয়ে filter; distance যত কম তত কাছে।
- Semantic synonym ধরে, keyword ID/কোডে নির্ভরযোগ্য; production-এ hybrid।
- Chunking-এ fixed বাক্য ভাঙে, recursive কম ভাঙে, overlap বর্ডারের তথ্য বাঁচায় — দাম: বেশি চাংক।

**চেষ্টা করুন:** (১) `overlap_chunks`-এ `overlap=0` ও `overlap=100` দিয়ে চাংক সংখ্যা ও key sentence-এর অবস্থা দেখুন। (২) §3.3-এর `docs`-এ এমন query বানান যেখানে semantic ও keyword দুটোই ব্যর্থ হয়। (৩) ১০০ / ৩০০ / ১৫০০ চাংক সাইজে নিজের দুটো প্রশ্নের best চাংক তুলনা করুন। (৪) `where={"chunk_index": {"$lt": 2}}` দিয়ে শুধু প্রথম দুই চাংকে খুঁজুন।